# Libraries management

In [ ]:
import os
import sys
import pandas as pd
## Check python version
import platform

sys.path.insert(0, os.getcwd())      # notebook folder: nbconvert and VS Code both start the kernel here
import auto_lib as al

In [ ]:
# print(sys.version)
# print(platform.python_version())
# print(al.__version__, pd.__version__)

# Config management

In [ ]:
instant_client_path = r"D:\oracle\instantclient_19_29"

al.init_oracle_client_once(instant_client_path)   # Thick mode, loaded once per kernel: safe to re-run this cell
env = al.load_env()                               # DWH_* from the .env at the repo root, never printed

# Declaration of variables

In [ ]:
sql_script = """
SELECT
    *
FROM
    crv_data.loutruong_supplier_byr_perf_di
WHERE
    1 = 1
    AND (
        sale_date BETWEEN ADD_MONTHS(TRUNC(SYSDATE - 1, 'YYYY'), -12) AND TRUNC(SYSDATE - 1)
    )
    AND supplier_code IN (
        SELECT
            supplier_code
        FROM
            omni_digimgr.loutruong_dim_supplier
    )
ORDER BY
    sale_date ASC,
    supplier_code ASC
"""
sheet_path = r"D:\OneDrive - Central Group\Stella's files - 1. HAND OVER\03. REPORT DAILY\09_supplier_tracker\Supplier_Performance_Tracker.xlsx"
sheet_name = 'byr_perf_raw_di'
header_aliases = {}   # query column -> header text used in the sheet (positional paste, names may differ)
allow_empty_result = False     # False = stop before Excel is touched when the query returns 0 rows (upstream not loaded yet)
fetch_arraysize = 10000        # rows per round trip when fetching from Oracle (default 100 = slow on 400k+ rows)

# --- Tuning knobs: add any of them to ExcelJob(...) as a keyword argument to change the default shown here ---
#   strict_header_check=False  True = stop the run when the Excel header row differs from the query columns
#   open_retries=5             attempts to open the workbook while OneDrive / another process still holds it
#   open_retry_wait=15         seconds between open attempts
#   load_wait=3                seconds to let Excel settle after the workbook is opened
#   paste_chunk_rows=50000     rows per COM call, keeps memory flat on very large pastes
#   refresh_after_paste=True   refresh queries / pivots once the new raw data is in the sheet
#   refresh_timeout=600        max seconds to wait for background queries to finish
#   refresh_settle_wait=2      seconds to let Excel settle after the refresh
#   excel_exit_timeout=120     seconds to wait for EXCEL.EXE to disappear after Quit before force-closing it
job = al.ExcelJob(sheet_path=sheet_path, sheet_name=sheet_name, header_aliases=header_aliases)

# 3. Connection controller

In [ ]:
df = al.fetch_dataframe(sql_script, arraysize=fetch_arraysize, allow_empty=allow_empty_result, env=env)

In [ ]:
# print(df.head(10).to_string())

# print(df.dtypes)

# Main logic: Open > Delete > Write > Refresh > Save > Close

In [ ]:
rows_processed = al.paste_to_sheet(df, job)